# Provider personality traits over time

For each personality trait, plot the strength metric over time (release date) for Anthropic,
Gemini and OpenAI models. Each provider is one line; CI bounds (95% from `strength_with_stats`)
are shown as a faded fill around the line.

Mirrors the data-loading approach used in `02_plotting_paper.ipynb` (final table cell).

In [ ]:
import importlib
import pathlib
from datetime import datetime

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd

import feedback_forensics as ff
import feedback_forensics.app.plotting.paper as paper_plot

importlib.reload(paper_plot)
importlib.reload(ff)

fig_save_path = pathlib.Path("./output/png/traits_over_time")
fig_save_path.mkdir(parents=True, exist_ok=True)

# Same data file used in the final table cell of 02_plotting_paper.ipynb
data_path = pathlib.Path(
    "/Users/arduin/main/repos/huggingface/ff-model-personality/data/v2/annotations/combined_ap.json"
)

cache = {}
dataset = ff.DatasetHandler(cache=cache)
dataset.add_data_from_path(data_path)

In [ ]:
# Models to compare. Provider labels match the user-facing brand;
# release dates are best-effort approximations and can be edited as needed.
MODELS = {
    # --- Anthropic --------------------------------------------------------
    "openrouter/anthropic/claude-3.7-sonnet":   ("Anthropic", "2025-02-24"),
    "openrouter/anthropic/claude-sonnet-4":     ("Anthropic", "2025-05-22"),
    "openrouter/anthropic/claude-sonnet-4.5":   ("Anthropic", "2025-09-29"),
    #"openrouter/anthropic/claude-haiku-4.5":    ("Anthropic", "2025-10-15"),
    "openrouter/anthropic/claude-opus-4.7":     ("Anthropic", "2026-04-16"),
    # --- Google (Gemini) --------------------------------------------------
    "openrouter/google/gemini-2.5-pro":          ("Gemini",   "2025-06-17"),
    "openrouter/google/gemini-3-pro-preview":    ("Gemini",   "2025-11-18"),
    "openrouter/google/gemini-3.1-pro-preview":  ("Gemini",   "2026-02-19"),
    # --- OpenAI -----------------------------------------------------------
    #"openrouter/openai/gpt-3.5-turbo":           ("OpenAI",   "2022-11-30"),
    #"openrouter/openai/gpt-4.1-mini":            ("OpenAI",   "2025-04-14"),
    #"openrouter/openai/gpt-5":                   ("OpenAI",   "2025-08-07"),
    "openrouter/openai/gpt-5-chat":              ("OpenAI",   "2025-08-07"),
    #"openrouter/openai/gpt-5.1":                 ("OpenAI",   "2025-11-12"),
    "openrouter/openai/gpt-5.1-chat":            ("OpenAI",   "2025-11-12"),
    "openrouter/openai/gpt-5.3-chat":            ("OpenAI",   "2026-03-03"),
}

# Reference models always have strength=0 by construction (they are the baseline
# the others are compared against). They have no CI to plot.
#
# GPT-4o-2024-11-20 was actually released on 2024-11-20, but to keep the x-axis
# compact we display it at the earliest non-reference model date (2025-02-24,
# Claude 3.7 Sonnet) and replace the corresponding x-tick label with
# REFERENCE_TICK_LABEL so the reader knows the reference is from that date or
# earlier.
REFERENCE_MODELS = {
    "openrouter/openai/gpt-4o-2024-11-20": ("OpenAI", "2025-02-24"),
}
REFERENCE_TICK_DATE = "2025-02-24"
REFERENCE_TICK_LABEL = "2025-02\nor earlier"

# Replace specific auto-located tick months with a different month. Used to
# nudge the AutoDateLocator output (e.g. move the mid-2025 tick to October).
# Keys/values are "YYYY-MM" strings; both are anchored to the 1st of the month.
TICK_REPLACEMENTS = {
    "2025-07": "2025-09",
}

PROVIDERS = ["Anthropic", "OpenAI", "Gemini"]
PROVIDER_COLORS = {
    "Anthropic": "#D97757",  # warm orange
    "OpenAI":    "#10A37F",  # green
    "Gemini":    "#4285F4",  # blue
}

# --- Plot styling (tweak these) ---------------------------------------------
# Single-trait plots (used by the per-trait loop)
FONT_SIZE = 18               # axis labels, ticks, legend
TITLE_SCALE = 1.1            # title font size = FONT_SIZE * TITLE_SCALE
MAX_X_TICKS = 4              # fewer x-axis ticks to declutter the plot

SINGLE_FIG_WIDTH = 9.0
SINGLE_FIG_HEIGHT = 5.5

# Three-panel highlight figure (sized for A4 portrait page with ~1" margins).
# Use paper-typical font sizes here — they are independent of FONT_SIZE.
ROW_FONT_SIZE = 9            # paper-style font size
ROW_TITLE_SCALE = 1.05       # title ≈ 9.5 pt
ROW_MAX_X_TICKS = 3
ROW_FIG_WIDTH = 6.3          # A4 text width
ROW_FIG_HEIGHT = 1.8
ROW_WSPACE = 0.08            # horizontal gap between panels (axes-fraction)

In [ ]:
# Pick the model-identity annotators corresponding to MODELS and compute strength_with_stats
annotator_metadata = dataset.get_available_annotators()

selected_annotators = {
    annotator_key: meta
    for annotator_key, meta in annotator_metadata.items()
    if meta.get("model_id") in MODELS
}

missing_models = sorted(set(MODELS) - {m["model_id"] for m in selected_annotators.values()})
if missing_models:
    print("Warning: these models are not present as annotators in the dataset:")
    for m in missing_models:
        print("  -", m)

dataset.set_annotator_cols(annotator_keys=list(selected_annotators.keys()))

# Map column name (annotator_visible_name) -> model_id so we can look up provider/date
col_to_model = {
    meta["annotator_visible_name"]: meta["model_id"]
    for meta in selected_annotators.values()
}

df = dataset.get_annotator_metrics_df(
    metric_name="strength_with_stats",
    index_col_name="Generate a response that...",
)
df.head()

In [ ]:
# Reshape into a long-form table: one row per (trait, model)
records = []
for _, row in df.iterrows():
    trait = row["Generate a response that..."]
    for col_name, cell in row.items():
        if col_name in ("Generate a response that...", "Max diff"):
            continue
        if not isinstance(cell, dict):
            continue
        model_id = col_to_model.get(col_name)
        if model_id is None or model_id not in MODELS:
            continue
        provider, date = MODELS[model_id]
        records.append(
            {
                "trait": trait,
                "provider": provider,
                "model_id": model_id,
                "model_short": model_id.split("/")[-1],
                "date": pd.Timestamp(date),
                "strength": cell.get("strength"),
                "ci_lower": cell.get("ci_lower_95"),
                "ci_upper": cell.get("ci_upper_95"),
                "p_value": cell.get("p_value"),
            }
        )

records_df = pd.DataFrame(records)
print(f"Traits: {records_df['trait'].nunique()} | rows: {len(records_df)}")
records_df.head()

# Inject reference-model rows: by definition strength = 0 (no CI) for every trait
ref_rows = []
for trait in records_df["trait"].unique():
    for ref_model_id, (ref_provider, ref_date) in REFERENCE_MODELS.items():
        ref_rows.append(
            {
                "trait": trait,
                "provider": ref_provider,
                "model_id": ref_model_id,
                "model_short": ref_model_id.split("/")[-1],
                "date": pd.Timestamp(ref_date),
                "strength": 0.0,
                "ci_lower": 0.0,
                "ci_upper": 0.0,
                "p_value": None,
                "is_reference": True,
            }
        )
records_df["is_reference"] = False
if ref_rows:
    records_df = pd.concat([pd.DataFrame(ref_rows), records_df], ignore_index=True)
print(f"After adding references: {records_df['trait'].nunique()} traits | {len(records_df)} rows")


In [ ]:
def plot_trait(
    trait_name,
    trait_df,
    save_dir=fig_save_path,
    show=True,
    ax=None,
    font_size=None,
    title_scale=None,
    max_x_ticks=None,
    display_title=None,
    paper_style=False,
):
    """Single-trait plot: strength over time per provider with CI fill.

    If ``ax`` is provided, draw onto it (used for multi-panel layouts). Otherwise
    create a standalone figure and save it.

    Font sizing/tick params can be overridden per call so the standalone plots
    and the multi-panel highlight figure can have independent styling.

    ``display_title`` overrides the rendered title (use to manually wrap long
    titles, e.g. with an embedded ``\n``).
    """
    fs = FONT_SIZE if font_size is None else font_size
    ts = TITLE_SCALE if title_scale is None else title_scale
    mt = MAX_X_TICKS if max_x_ticks is None else max_x_ticks

    if ax is None:
        fig, ax = plt.subplots(figsize=(SINGLE_FIG_WIDTH, SINGLE_FIG_HEIGHT))
        standalone = True
    else:
        fig = ax.figure
        standalone = False

    for provider in PROVIDERS:
        sub = (
            trait_df[trait_df["provider"] == provider]
            .dropna(subset=["strength"])
            .sort_values("date")
        )
        if sub.empty:
            continue
        color = PROVIDER_COLORS[provider]

        # CI fill (faded). Reference rows have ci_lower == ci_upper == 0, so the
        # band naturally pinches to zero at that point — that's correct since the
        # reference model has no uncertainty in its own (zero) strength.
        if sub[["ci_lower", "ci_upper"]].notna().all().all():
            ax.fill_between(
                sub["date"],
                sub["ci_lower"],
                sub["ci_upper"],
                color=color,
                alpha=0.18,
                linewidth=0,
            )

        # Main line
        ax.plot(
            sub["date"],
            sub["strength"],
            marker="o",
            color=color,
            linewidth=1.8,
            markersize=5,
            label=provider,
        )

        # Highlight reference markers with a star on top
        if "is_reference" in sub.columns:
            ref_pts = sub[sub["is_reference"]]
            if not ref_pts.empty:
                ax.scatter(
                    ref_pts["date"],
                    ref_pts["strength"],
                    marker="*",
                    s=max(60, fs * 8),
                    color=color,
                    edgecolor="black",
                    linewidth=0.6,
                    zorder=5,
                )

    ax.axhline(0, color="grey", linewidth=0.8, linestyle=":")
    title_text = trait_name if display_title is None else display_title
    ax.set_title(title_text, fontsize=fs * ts, fontstyle="italic")
    ax.set_xlabel("Release date", fontsize=fs)
    ax.set_ylabel("Strength", fontsize=fs)
    ax.tick_params(axis="both", which="major", labelsize=fs)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", frameon=True, fontsize=fs)

    # Paper-style tick cleanup: drop minor ticks and remove top/right ticks
    # entirely. Only applied when paper_style=True (e.g. for the multi-panel
    # highlight figure); single-trait plots keep matplotlib defaults.
    if paper_style:
        ax.minorticks_off()
        ax.tick_params(which="both", top=False, right=False)

    # Date formatting / fewer ticks on x-axis. If a reference point is present,
    # force its date to be the leftmost tick and label it with REFERENCE_TICK_LABEL
    # (e.g. "(2025-02 or earlier)") so the reader knows the reference predates
    # the rest of the series.
    ref_num = None
    if "is_reference" in trait_df.columns and trait_df["is_reference"].any():
        ref_dates = trait_df.loc[trait_df["is_reference"], "date"]
        if not ref_dates.empty:
            ref_num = mdates.date2num(ref_dates.iloc[0])

    ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=mt, minticks=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

    if ref_num is not None:
        # Compute auto ticks, drop any at/before the reference, then prepend ref
        fig.canvas.draw()
        auto_ticks = sorted(set(ax.get_xticks()))
        ticks = [t for t in auto_ticks if t > ref_num + 5]

        # Apply any user-configured month replacements (e.g. 2025-07 -> 2025-10)
        replaced = []
        for t in ticks:
            key = mdates.num2date(t).strftime("%Y-%m")
            if key in TICK_REPLACEMENTS:
                target = pd.Timestamp(TICK_REPLACEMENTS[key] + "-01")
                replaced.append(mdates.date2num(target))
            else:
                replaced.append(t)
        ticks = sorted(set(replaced))

        ticks = [ref_num] + ticks
        ax.xaxis.set_major_locator(mtick.FixedLocator(ticks))

        def _date_fmt(x, pos, _ref=ref_num):
            if abs(x - _ref) < 1:
                return REFERENCE_TICK_LABEL
            return mdates.num2date(x).strftime("%Y-%m")
        ax.xaxis.set_major_formatter(mtick.FuncFormatter(_date_fmt))

    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

    if standalone:
        plt.tight_layout()
        safe_name = "".join(c if c.isalnum() else "_" for c in str(trait_name)).strip("_")[:90]
        out_path = pathlib.Path(save_dir) / f"trait_{safe_name}.png"
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        if show:
            plt.show()
        else:
            plt.close(fig)
        return out_path
    return None


In [ ]:
# Optional: rank traits by max difference across providers (largest signal first)
trait_order = (
    records_df.groupby("trait")["strength"]
    .agg(lambda s: s.max() - s.min())
    .sort_values(ascending=False)
    .index.tolist()
)

# Set TOP_N to a number to limit output, or None to plot every trait
TOP_N = None
traits_to_plot = trait_order if TOP_N is None else trait_order[:TOP_N]

saved_paths = []
for trait in traits_to_plot:
    trait_df = records_df[records_df["trait"] == trait]
    saved_paths.append(plot_trait(trait, trait_df))

print(f"Saved {len(saved_paths)} plots to {fig_save_path}")

In [ ]:
# Highlight figure: three traits side-by-side, sharing the strength (y) axis.
# Paper-style font sizes; sized to fit the text width of an A4 portrait page.
HIGHLIGHT_TRAITS = [
    "uses more casual language",
    "uses more bold and italics text",
    "compliments the user's question or prompt",
]

# Manual two-line wrapping for titles that don't fit on one line at this width
TITLE_OVERRIDES = {
    "compliments the user's question or prompt": "compliments the user's\nquestion or prompt",
}

# Slightly taller to accommodate the wrapped two-line title without squashing axes
fig, axes = plt.subplots(
    1, len(HIGHLIGHT_TRAITS),
    figsize=(ROW_FIG_WIDTH, ROW_FIG_HEIGHT + 0.25),
    sharey=True,
)
if len(HIGHLIGHT_TRAITS) == 1:
    axes = [axes]

for ax, trait in zip(axes, HIGHLIGHT_TRAITS):
    trait_df = records_df[records_df["trait"] == trait]
    if trait_df.empty:
        ax.set_visible(False)
        print(f"Trait not found: {trait!r}")
        continue
    plot_trait(
        trait, trait_df, ax=ax,
        font_size=ROW_FONT_SIZE,
        title_scale=ROW_TITLE_SCALE,
        max_x_ticks=ROW_MAX_X_TICKS,
        display_title=TITLE_OVERRIDES.get(trait),
        paper_style=True,
    )

# Only the leftmost panel keeps an ylabel; others share the y axis
for ax in axes[1:]:
    ax.set_ylabel("")

# Only the center panel keeps the "Release date" xlabel
center_idx = len(axes) // 2
for i, ax in enumerate(axes):
    if i != center_idx:
        ax.set_xlabel("")

# Build a single legend for the whole figure (dedup handles across panels)
handles_all, labels_all = [], []
for ax in axes:
    h, l = ax.get_legend_handles_labels()
    handles_all += h
    labels_all += l
seen, h_dedup, l_dedup = set(), [], []
for hh, ll in zip(handles_all, labels_all):
    if ll not in seen:
        seen.add(ll); h_dedup.append(hh); l_dedup.append(ll)
for ax in axes:
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()

# Tighten horizontal spacing between subplots, then place the legend above.
plt.subplots_adjust(left=0.07, right=0.99, top=0.78, bottom=0.20, wspace=ROW_WSPACE)

fig.legend(
    h_dedup, l_dedup,
    loc="upper center", ncol=len(l_dedup),
    frameon=True, fontsize=ROW_FONT_SIZE,
    bbox_to_anchor=(0.5, 1.0),
)

highlight_path = fig_save_path / "highlight_three_traits.png"
plt.savefig(highlight_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {highlight_path}")
